# Laboratorio de regresión - 2

## Significancia de factores

|                |   |
:----------------|---|
| **Nombre**     |  David Farias Rivera |
| **Fecha**      |  16/10/2026 |
| **Expediente** |  754381 |

Descarga el archivo de publicidad y carga los datos (Advertising.csv).

In [35]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
from sklearn.linear_model import LinearRegression  # El modelo de regresion
from scipy import stats  # Lo usamos al final para sacar los p-values


data = pd.read_csv("/Users/david/Documents/Codigod/ITESO/01-ITESO/Lab 3/Advertising.csv", index_col=0, header=0)
data.head()

,TV,radio,newspaper,sales
1,230.1,37.8,69.2,22.1
2,44.5,39.3,45.1,10.4
3,17.2,45.9,69.3,9.3
4,151.5,41.3,58.5,18.5
5,180.8,10.8,58.4,12.9


**¿Hay alguna relación entre el presupuesto para publicidad y las ventas?**

Nuestra primera meta debe ser determinar si hay evidencia en los datos de que haya una asociación entre estas variables.

- ¿Por qué? ¿Qué resultaría si nos diéramos cuenta de la falta de relación entre el presupuesto de publicidad y las ventas?

**- Primero hay que ver si hay relación, porque si no la hay no tiene caso preguntarnos lo demás. Si no hubiera relación, el dinero en publicidad no se estaría reflejando en ventas y convendría usarlo en otra cosa.**

**¿Qué tan fuerte es esta relación?**
Asumiendo que existe esta relación, ¿nos sirve conocer el impacto que tiene invertir en publicidad en las ventas?

**- Sí sirve. No basta con saber que hay relación, también importa qué tan grande es, porque eso nos dice si lo que vendemos de más alcanza a pagar lo que gastamos.**



**¿Cuáles medios están asociados con las ventas? ¿Qué tan grande es la asociación entre un medio específico y las ventas?**

Hay 3 medios distintos en los datos. ¿Sirve invertir en los 3? ¿Conviene más invertir sólo en uno?

**¿Qué tan seguros estamos de que podríamos predecir ventas futuras?**

**¿La relación es lineal?**

**¿Hay sinergia entre estos medios?**

Puede ser que gastar \\$50,000 en publicidad y otros \\$50,000 en radio es mejor opción que gastar \\$100,000 en televisión. A esto le llamamos efecto de interacción.

Podemos usar regresión lineal para responder todas estas preguntas.

Realiza una regresión lineal:

$$ \text{ventas} \approx \beta_0 + (\beta_1)(\text{TV})$$

In [36]:
TV = data[["TV"]]
sales = data[["sales"]]
lr = LinearRegression()  # Creamos el modelo
lr.fit(TV, sales)  


b0 = lr.intercept_ 
b1 = lr.coef_[0]
b0, b1

(array([7.03259355]), array([0.04753664]))

### Verificando la precisión de nuestros coeficientes estimados

Recuerda que en el mundo real hay ruidos y errores de medición. Siempre se asume que la verdadera relación entre $X$ y $Y$ es $$Y = \beta_0 + \beta_1 X + \epsilon$$

Se asume que el término de error es independiente de $X$ (el error siempre es el mismo sin importar el valor de $X$). Este modelo describe a la *línea de regresión de la población*, que es la mejor aproximación de la verdadera relación entre $X$ y $Y$. Cuando usamos mínimos cuadrados encontramos la *línea de mínimos cuadrados*.

¿Cuál es la diferencia entre población y muestra?

- **La poblacion es la totalidad de los datos y la muestra solo la parte sobre la que etamos trabajando o tenemos mediciones**

¿Cuál crees que sea la diferencia entre hacer una regresión con todos los datos de la población y una muestra de ella?

- **Pues una regresion con todos los datos tendriamos una funcion y un modelo mucho mas preciso pero mas lento y tardado, a diferencia de que con muestra es una aproximacion**

La línea de regresión de la población no se puede observar. El concepto de comparar estas líneas es una extensión natural del acercamiento estadístico estándar de usar información de una muestra para estimar características de una población grande.

Imagina que quieres encontrar la altura promedio de un mexicano $\mu$. Medir a todos y cada uno de los mexicanos en situaciones similares, con la misma regla, mismo operador, y otras incontables formas de minimizar la variación de la medida es una tarea imposible. Lo que podemos asumir es que $\hat{\mu} = \bar{y}$. La media poblacional y la media muestral son diferentes, pero la media muestral es usualmente un buen estimado.

De la misma manera, como no contamos con el 100% de la información para hacer una regresión, los coeficientes $\beta_0$ y $\beta_1$ son desconocidos. Podemos estimarlos usando mínimos cuadrados, encontrando $\hat{\beta_0}$ y $\hat{\beta_1}$. Puede que las muestras que tengamos en ese momento estén un poco por encima de la media, pero otras muestras en otro momento puede que estén debajo de la media. En general, esperamos que el promedio de las aproximaciones $\hat{\mu}$ aproxime a $\mu$.

Esto lleva a la pregunta: ¿qué tan cercanos son nuestros coeficientes estimados a los verdaderos coeficientes? Utilizamos el concepto de error estándar para evaluar esto.

$$ \text{Var}(\hat{\mu})=\text{SE}(\hat{\mu})^2 = \frac{\sigma^2}{n} $$

Donde $\sigma$ es la desviación estándar de cada una de las observaciones $y_i$ de $Y$. El error estándar nos dice la cantidad promedio que el estimado difiere del valor verdadero. Podemos ver en la fórmula que entre más observaciones tengamos el error se hace más pequeño. Las fórmulas para errores estándar de $\hat{\beta_0}$ y $\hat{\beta_1}$ son:

$$ \text{SE}(\hat{\beta_0})^2 = \sigma^2 [\frac{1}{n} + \frac{\bar{x}^2}{\sum_{i=1}^n (x_i - \bar{x})^2}]$$

$$ \text{SE}(\hat{\beta_1})^2 = \frac{\sigma^2}{\sum_{i=1}^n (x_i - \bar{x})^2}$$

$$ \sigma^2 = \text{Var}(\epsilon) = \text{RSE}^2 = \frac{\text{RSS}}{n-p}$$

Para que estas fórmulas sean validas asumimos que los errores $\epsilon_i$ tienen varianza común $\sigma^2$ y que no están correlacionados.

Calcula los errores estándar de los coeficientes

In [37]:
n = len(sales)  # Numero de observaciones
p = 2  # Coeficientes que estimamos, b0 y b1

y_pred = lr.predict(TV)  # Lo que predice el modelo
rss = np.sum((sales - y_pred) ** 2)  # Residuos al cuadrado
sigma2 = rss / (n - p)  # Varianza del error

x_m = np.mean(data["TV"])  # Promedio de TV
sxx = np.sum((data["TV"] - x_m) ** 2)  # Que tanto se mueve TV alrededor de su promedio

error_b0 = np.sqrt(sigma2 * (1 / n + x_m ** 2 / sxx))  # Error estandar de b0
error_b1 = np.sqrt(sigma2 / sxx)  # Error estandar de b1
error_b0, error_b1


/opt/anaconda3/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:84: FutureWarning: The behavior of DataFrame.sum with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return reduction(axis=axis, out=out, **passkwargs)


(sales    0.457843
 dtype: float64,
 sales    0.002691
 dtype: float64)

Estos errores se pueden usar para calcular intervalos de confianza. Un intervalo de confianza del $95\%$ se define como un rango de valores en el cuál se encuentra el desconocido valor verdadero con un $95\%$ de probabilidad.

Otra forma de verlo es que si tomamos muestras repetidas y construimos un intervalo de confianza para cada una, el $95\%$ de los intervalos creados van a contener el valor verdadero. Para la regresión el intervalo de confianza del $95\%$ toma la forma:

$$ \hat{\beta_j} \pm 2\text{SE}(\hat{\beta_j})$$

Calcula los intervalos de confianza para los coeficientes estimados:

In [38]:
b0_inf = b0 - 2 * error_b0  # Limite de abajo de b0
b0_sup = b0 + 2 * error_b0  # Limite de arriba de b0
b1_inf = b1 - 2 * error_b1  # Limite de abajo de b1
b1_sup = b1 + 2 * error_b1  # Limite de arriba de b1
b0_inf, b0_sup, b1_inf, b1_sup


(sales    6.116908
 dtype: float64,
 sales    7.948279
 dtype: float64,
 sales    0.042155
 dtype: float64,
 sales    0.052918
 dtype: float64)

Los errores estándar también se usan para realizar pruebas de hipótesis. La prueba de hipótesis más común es probar la hipótesis nula de:

$$ H_0: \text{No hay relación entre } X \text{ y } Y \ \ \ \ (\beta_1=0)$$

contra la hipótesis alternativa:
$$ H_0: \text{Hay alguna relación entre } X \text{ y } Y \ \ \ (\beta_1 \neq 0)$$

Explica con tus palabras el significado de la hipótesis nula y la hipótesis alternativa.

- **La hipótesis nula dice que la TV no tiene nada que ver con las ventas (b1 = 0) y su alternativa es que si lo tenga**

Para probal la hipótesis nula debemos determinar si nuestro estimado $\hat{\beta_1}$ de $\beta_1$ está lo suficientemente alejado de cero para que podamos decir con confianza que este valor no es cero. 

¿Qué tan lejos? Depende de qué tanta confianza tengamos en el estimado encontrado. Si nuestro error estándar es pequeño y nuestro estimado está alejado de cero podríamos decir que hay muy poca probabilidad de que el valor verdadero sea 0. En cambio, si nuestro error estándar es grande y nuestro estimado está muy cerca de cero, entonces podrías ser que el valor verdadero sea cero y que no haya relación entre las variables.

Se calcula un *estadístico t* dado por
$$ t = \frac{\hat{\beta_j} - \mu}{\text{SE}(\hat{\beta_j})} $$

donde $\mu$ es el valor contra el que queremos probar.

Calcula el estadístico t para tus coeficientes estimados, usando como referencia la prueba de hipótesis.

In [39]:
t_b0 = (b0 - 0) / se_b0  # Que tan lejos de cero esta b0
t_b1 = (b1 - 0) / se_b1  # Que tan lejos de cero esta b1
t_b0, t_b1

(sales    15.360275
 dtype: float64,
 sales    17.667626
 dtype: float64)

La distribución t tiene forma de campana y se parece bastante a la distribución normal cuando $n > 30$. Ya sólo es cuestión de calcular la probabilidad de observar cualquier número tal que su valor absoluto sea igual o mayor que el valor absoluto del estadístico t calculado. En otras palabras:
$$ P(|x| \geq |t|) $$

A esta probabilidad la llamamos *p-value*. Un *p-value* pequeño indica que es poco probable que exista por puro azar una relación significativa entre predictor y respuesta, en caso de que no haya una asociación real entre predictor y respuesta. En otras palabras, el *p-value* te dice la probabilidad de que parezca que hay relación cuando no la hay.

Si el *p-value* es pequeño, inferimos que sí hay una asociación entre el predictor y la respuesta, y **rechazamos la hipótesis nula**.
  

¿Qué tan pequeño? Depende de la aplicación. Un valor muy común es del $5\%$.

Utiliza el siguiente código para calcular el *p-value* para tus coeficientes

`from scipy import stats`

`p_bj = 2*(1 - stats.t.cdf(np.abs(t_bj), n-p))`

In [40]:
p_b0 = 2 * (1 - stats.t.cdf(np.abs(t_b0), n - p))  # p-value de b0
p_b1 = 2 * (1 - stats.t.cdf(np.abs(t_b1), n - p))  # p-value de b1
print(p_b0, p_b1)


[0.] [0.]


¿Se rechaza la hipótesis nula? ¿Qué significa?

- **Sí se rechaza, porque el p-value es mucho menor a 5%. Quiere decir que sí hay relación entre el gasto en TV y las ventas.**

Realiza otras dos regresiones. Ya tienes hecha la regresión de ventas dado el gasto en publicidad de TV. Realiza la regresión para gastos en radio y gastos en periódico. Organiza las respuestas para que debajo de esta celda se tenga:
- Título de regresión
- Coeficientes estimados
- Errores estándar de los coeficientes
- Intervalos de confianza
- Estadísticos t
- p-values
- Observaciones

In [45]:
t_crit = 2  # Cuantos errores estandar nos alejamos para el 95%

# Radio
radio = data[["radio"]]  # Gasto en radio, doble corchete porque sklearn lo quiere como tabla
lr = LinearRegression()  # Creamos el modelo
lr.fit(radio, sales)  # Lo entrenamos
b0_r = lr.intercept_  # Intercepto
b1_r = lr.coef_[0]  # Pendiente

y_pred = lr.predict(radio)  # Lo que predice el modelo
rss = np.sum((sales - y_pred) ** 2)  # Residuos al cuadrado
sigma2 = rss / (n - p)  # Varianza del error
x_m = np.mean(data["radio"])  # Promedio de radio
sxx = np.sum((data["radio"] - x_m) ** 2)  # Que tanto se mueve radio alrededor de su promedio

error_b0_r = np.sqrt(sigma2 * (1 / n + x_m ** 2 / sxx))  # Error estandar de b0
error_b1_r = np.sqrt(sigma2 / sxx)  # Error estandar de b1

b0_inf_r = b0_r - t_crit * error_b0_r  # Limite de abajo de b0
b0_sup_r = b0_r + t_crit * error_b0_r  # Limite de arriba de b0
b1_inf_r = b1_r - t_crit * error_b1_r  # Limite de abajo de b1
b1_sup_r = b1_r + t_crit * error_b1_r  # Limite de arriba de b1

t_b0_r = (b0_r - 0) / error_b0_r  # Que tan lejos de cero esta b0
t_b1_r = (b1_r - 0) / error_b1_r  # Que tan lejos de cero esta b1

p_b0_r = 2 * (1 - stats.t.cdf(np.abs(t_b0_r), n - p))  # p-value de b0
p_b1_r = 2 * (1 - stats.t.cdf(np.abs(t_b1_r), n - p))  # p-value de b1

# Periodico
newspaper = data[["newspaper"]]  # Gasto en periodico
lr = LinearRegression()  # Creamos otro modelo
lr.fit(newspaper, sales)  # Lo entrenamos
b0_n = lr.intercept_  # Intercepto
b1_n = lr.coef_[0]  # Pendiente

y_pred = lr.predict(newspaper)  # Lo que predice el modelo
rss = np.sum((sales - y_pred) ** 2)  # Residuos al cuadrado
sigma2 = rss / (n - p)  # Varianza del error
x_m = np.mean(data["newspaper"])  # Promedio de periodico
sxx = np.sum((data["newspaper"] - x_m) ** 2)  # Que tanto se mueve periodico alrededor de su promedio

error_b0_n = np.sqrt(sigma2 * (1 / n + x_m ** 2 / sxx))  # Error estandar de b0
error_b1_n = np.sqrt(sigma2 / sxx)  # Error estandar de b1

b0_inf_n = b0_n - t_crit * error_b0_n  # Limite de abajo de b0
b0_sup_n = b0_n + t_crit * error_b0_n  # Limite de arriba de b0
b1_inf_n = b1_n - t_crit * error_b1_n  # Limite de abajo de b1
b1_sup_n = b1_n + t_crit * error_b1_n  # Limite de arriba de b1

t_b0_n = (b0_n - 0) / error_b0_n  # Que tan lejos de cero esta b0
t_b1_n = (b1_n - 0) / error_b1_n  # Que tan lejos de cero esta b1

p_b0_n = 2 * (1 - stats.t.cdf(np.abs(t_b0_n), n - p))  # p-value de b0
p_b1_n = 2 * (1 - stats.t.cdf(np.abs(t_b1_n), n - p))  # p-value de b1

# Resultados
print("Regresion de ventas con radio")  # Titulo
print("Coeficientes:", b0_r, b1_r)  # b0 y b1
print("Errores estandar:", error_b0_r, error_b1_r)  # Error estandar de cada uno
print("IC b0:", b0_inf_r, "a", b0_sup_r)  # Intervalo de b0
print("IC b1:", b1_inf_r, "a", b1_sup_r)  # Intervalo de b1
print("t:", t_b0_r, t_b1_r)  # Estadisticos t
print("p-values:", p_b0_r, p_b1_r)  # p-values
print()  # Espacio entre las dos regresiones

print("Regresion de ventas con periodico")  # Titulo
print("Coeficientes:", b0_n, b1_n)  # b0 y b1
print("Errores estandar:", error_b0_n, error_b1_n)  # Error estandar de cada uno
print("IC b0:", b0_inf_n, "a", b0_sup_n)  # Intervalo de b0
print("IC b1:", b1_inf_n, "a", b1_sup_n)  # Intervalo de b1
print("t:", t_b0_n, t_b1_n)  # Estadisticos t
print("p-values:", p_b0_n, p_b1_n)  # p-values


Regresion de ventas con radio
Coeficientes: [9.3116381] [0.20249578]
Errores estandar: sales    0.5629
dtype: float64 sales    0.020411
dtype: float64
IC b0: sales    8.185837
dtype: float64 a sales    10.437439
dtype: float64
IC b1: sales    0.161673
dtype: float64 a sales    0.243318
dtype: float64
t: sales    16.542245
dtype: float64 sales    9.920765
dtype: float64
p-values: [0.] [0.]

Regresion de ventas con periodico
Coeficientes: [12.35140707] [0.0546931]
Errores estandar: sales    0.62142
dtype: float64 sales    0.016576
dtype: float64
IC b0: sales    11.108567
dtype: float64 a sales    13.594247
dtype: float64
IC b1: sales    0.021542
dtype: float64 a sales    0.087845
dtype: float64
t: sales    19.876096
dtype: float64 sales    3.299591
dtype: float64
p-values: [0.] [0.0011482]


/opt/anaconda3/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:84: FutureWarning: The behavior of DataFrame.sum with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return reduction(axis=axis, out=out, **passkwargs)
/opt/anaconda3/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:84: FutureWarning: The behavior of DataFrame.sum with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return reduction(axis=axis, out=out, **passkwargs)


## Regresión lineal múltiple

En lugar de hacer una regresión para cada factor independiente, quizás se puede extender el modelo para que tenga varios factores dentro:

$$ Y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + ... + \beta_p X_p + \epsilon $$

Para nuestro ejemplo de publicidad:

$$ \text{sales} = \beta_0 + \beta_1 (\text{TV}) + \beta_2 (\text{radio}) + \beta_3 (\text{newspaper}) + \epsilon $$

Utiliza la librería `statsmodels` para realizar la regresión. Por defecto la librería statsmodels no toma en cuenta el intercepto ($\beta_0$), por lo que se tendrá que agregar una columna de unos de tamaño *n* a la matriz X.

`import statsmodels.api as sm`

`ols = sm.OLS(Y, X)`

`results = ols.fit()`

`results.summary()`

In [46]:
import statsmodels.api as sm  # Libreria para la parte estadistica

x_3 = data[["TV", "radio", "newspaper"]]  # Los tres medios juntos
x_3["unos"] = np.ones(n)  # Columna de unos de tamaño n para el intercepto
ols = sm.OLS(sales, x_3)  # Armamos el modelo
results = ols.fit()  # Lo entrenamos
results.summary()  # Vemos todo junto


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  sales   R-squared:                       0.897
Model:                            OLS   Adj. R-squared:                  0.896
Method:                 Least Squares   F-statistic:                     570.3
Date:                Fri, 16 Oct 2026   Prob (F-statistic):           1.58e-96
Time:                        04:21:43   Log-Likelihood:                -386.18
No. Observations:                 200   AIC:                             780.4
Df Residuals:                     196   BIC:                             793.6
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
TV             0.0458      0.001     32.809      0.000       0.043       0.049
radio          0.1885      0.009     21.893      0.000       0.172       0.206
newspaper     -0.0010      0.006     -0.177      0.860      -0.013       0.011
unos           2.9389      0.312      9.422      0.000       2.324       3.554
==============================================================================
Omnibus:                       60.414   Durbin-Watson:                   2.084
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              151.241
Skew:                          -1.327   Prob(JB):                     1.44e-33
Kurtosis:                       6.332   Cond. No.                         454.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

¿Qué diferencias puedes observar entre los *p-values* de una regresión múltiple y los encontrados en las regresiones simples? ¿Por qué crees que existen estas diferencias?

- **En las simples el periódico salía significativo y en la múltiple ya no (p de 0.86). Esto pasa porque en la múltiple cada coeficiente se mide manteniendo los otros medios constantes. El periódico está relacionado con la radio, entonces solo se llevaba el crédito de lo que hacía la radio.**

## Referencia

James, G., Witten, D., Hastie, T., Tibshirani, R.,, Taylor, J. (2023). An Introduction to Statistical Learning with Applications in Python. Cham: Springer. ISBN: 978-3-031-38746-3

## Addendum

Para calcular los *p-values* de los parámetros sin `statsmodels`:
1. Calcular RSS
2. Calcular RSE
3. `var_beta = np.linalg.inv(X.T @ X) * rse**2` (X con columna de intercepto)
4. `std_beta = np.sqrt(var_beta.diagonal())` El orden de los valores corresponde al orden de los factores en las columnas de la matriz $X$.
5. Calcular *estadístico t*
6. Calcular *p-value*